In [16]:
import os, numpy as np, matplotlib.pyplot as plt
from helper_functions.A_loading_helpers import load_csv_data
import helper_functions.C_preprocessing_helpers as ph  # alias pratique



In [17]:
Xtr, Xte, ytr, id_tr, id_te = load_csv_data("data")

In [18]:
# chemins (ceux de Jakob)
classes_path = "data/feature_properties/feature_classes.json"
feature_names_path = "data/feature_properties/feature_names.csv"

fd = ph.build_feature_dictionary(classes_path, feature_names_path)
feature_names = ph.load_feature_names(feature_names_path)

# continues “normales” + continues où la valeur max = “missing”
idx_cont = fd.get("continuous", {}).get("indices", [])
idx_cont_max_is_na = fd.get("continuous_but_null_also_a_number", {}).get("indices", [])
idx_all = sorted(set(idx_cont + idx_cont_max_is_na))

In [19]:
os.makedirs("reports/figs/boxplots", exist_ok=True)

def _col_clean(X, j, treat_max_as_nan):
    """
    Return a cleaned 1-D copy of column j from matrix X.

    - Casts to float.
    - If `treat_max_as_nan` is True, replaces the column's maximum value
      (often used as a sentinel for "no answer") with NaN.
    - Drops NaNs so plotting/stats work without errors.

    Parameters
    ----------
    X : np.ndarray
        2-D array of shape (n_samples, n_features). This is typically
        pre-filtered by class, e.g., Xtr[ytr == -1].
    j : int
        Feature index (column) to extract and clean.
    treat_max_as_nan : bool
        If True, treat the maximum observed value as missing and remove it.

    Returns
    -------
    np.ndarray
        1-D array containing the cleaned values for column j.
    """
    col = X[:, j].astype(float)
    if treat_max_as_nan and col.size:
        m = np.nanmax(col)
        col = np.where(col == m, np.nan, col)
    return col[~np.isnan(col)]

def boxplot_one_feature(j):
    """
    Create and save a per-class boxplot for feature j.

    Uses globals:
      - Xtr, ytr          : training matrix and labels
      - feature_names     : list of human-readable feature names
      - idx_cont_max_is_na: indices where "max value" means missing

    Steps:
      1) Clean the column j separately for class -1 and class +1.
      2) If both are empty, skip plotting.
      3) Draw a two-box boxplot: left = class -1, right = class +1.
      4) Save the figure to reports/figs/boxplots/ as a PNG.
    """
    name = feature_names[j] if j < len(feature_names) else f"f{j}"
    a = _col_clean(Xtr[ytr==-1], j, j in idx_cont_max_is_na)
    b = _col_clean(Xtr[ytr== 1], j, j in idx_cont_max_is_na)
    if a.size==0 and b.size==0:  # rien à tracer
        return
    plt.figure()
    plt.boxplot([a, b], labels=['-1','1'], showfliers=True, whis=1.5)
    plt.title(f"{name} — boxplot by class")
    plt.xlabel("Classe"); plt.ylabel("Valeur")
    fname = f"reports/figs/boxplots/{j}_{name.replace('/','-')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight"); plt.close()

# Générer tous les boxplots (une image par feature continue)
for j in idx_all:
    boxplot_one_feature(j)

In [22]:
# 0) Output folder + clean previous exports
SAVE_DIR = "reports/figs/barplots"
os.makedirs(SAVE_DIR, exist_ok=True)

# Efface les anciennes figures pour éviter les doublons entre runs
for ext in ("png", "pdf", "svg"):
    for f in glob.glob(os.path.join(SAVE_DIR, f"bar_*.{ext}")):
        try:
            os.remove(f)
        except FileNotFoundError:
            pass

# Par sécurité, on ferme toute figure Matplotlib encore ouverte
plt.close('all')

# 2) Build explicit index list: categorical + ordinal ONLY
idx_cat = fd.get("categorical", {}).get("indices", [])
idx_ord = fd.get("ordinal", {}).get("indices", [])
indices_to_plot = sorted(set(idx_cat + idx_ord))

# 3) Detect categorical/ordinal groups that encode "no answer" with the MAX value
treat_max_as_na = set()
for key, payload in fd.items():
    key_low = key.lower()
    if (("categorical" in key_low) or ("ordinal" in key_low)) and (("max" in key_low) or ("null" in key_low)):
        treat_max_as_na.update(payload.get("indices", []))

# ---- Plot function (counts only, not per class) with safety for many categories ----

DPI = 150
PX_LIMIT = 2**16 - 1                  # Matplotlib per-dimension hard limit
INCH_LIMIT = PX_LIMIT / DPI           # ~436 inches at 150 DPI
SLOT_WIDTH_IN = 0.5                   # horizontal inches per category
MAX_BARS_TECH = int(INCH_LIMIT / SLOT_WIDTH_IN)  # ~873 categories
READABILITY_CAP = 80                  # human-friendly cap

def barplot_counts_cat_ord(j,
                           extra_missing_codes={7, 9, 97, 99},
                           save_dir=SAVE_DIR):
    """
    Make a bar plot of category COUNTS for feature j (categorical/ordinal).
    - Cleans: drops NaNs, common survey missing codes, and optional "max=NA".
    - If too many categories: plot Top-K and bucket the rest into 'Other'.
    - Skips ID-like columns (>=90% unique values).
    """
    name = feature_names[j] if j < len(feature_names) else f"f{j}"
    col = Xtr[:, j].astype(float)

    # Valid mask
    mask = ~np.isnan(col)

    # Handle "max value == missing"
    if j in treat_max_as_na and col.size:
        maxv = np.nanmax(col)
        mask &= (col != maxv)

    # Drop explicit survey missing codes (adjust if your dataset differs)
    if extra_missing_codes:
        mask &= ~np.isin(col, list(extra_missing_codes))

    x = col[mask]
    if x.size == 0:
        print(f"Skip {j} ({name}): all values missing/filtered.")
        return

    # Unique categories and counts (sorted by frequency desc)
    vals, counts = np.unique(x, return_counts=True)
    order = np.argsort(counts)[::-1]
    vals, counts = vals[order], counts[order]

    # Skip likely IDs (bar plot not meaningful)
    n = x.size
    if len(vals) / n >= 0.90:
        print(f"Skip {j} ({name}): looks like an ID (unique ratio >= 0.90).")
        return

    # Decide how many categories to show
    max_allowed = min(MAX_BARS_TECH, READABILITY_CAP)
    use_other = False
    if len(vals) > max_allowed:
        top_k = max_allowed - 1  # leave room for 'Other'
        top_vals = vals[:top_k]
        top_counts = counts[:top_k]
        other_count = counts[top_k:].sum()
        vals_plot = np.concatenate([top_vals, np.array(["Other"], dtype=object)])
        counts_plot = np.concatenate([top_counts, np.array([other_count])])
        use_other = True
    else:
        vals_plot = vals
        counts_plot = counts

    # Dynamic width but under pixel limits
    fig_w = min(INCH_LIMIT, max(6.0, SLOT_WIDTH_IN * len(vals_plot)))

    # Nice tick labels for integer-like categories (e.g., 1.0 -> "1")
    def _fmt_tick(v):
        try:
            return str(int(v)) if float(v).is_integer() else str(v)
        except Exception:
            return str(v)

    x_idx = np.arange(len(vals_plot))
    plt.figure(figsize=(fig_w, 4.2), dpi=DPI)
    plt.bar(x_idx, counts_plot, width=0.8)
    plt.xticks(x_idx, [_fmt_tick(v) for v in vals_plot], rotation=30, ha="right")
    plt.ylabel("Count")
    suffix = " (Top-K + Other)" if use_other else ""
    plt.title(f"{name} — category counts{suffix}")

    os.makedirs(save_dir, exist_ok=True)
    out = os.path.join(save_dir, f"bar_{j}_{name.replace('/', '-')}.png")
    plt.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.close()

# 4) Generate bar plots for ALL categorical + ordinal features
for j in indices_to_plot:
    try:
        barplot_counts_cat_ord(j)
    except Exception as e:
        print(f"Skip {j} ({feature_names[j] if j < len(feature_names) else j}): {e}")

print(f"Done: {len(indices_to_plot)} categorical/ordinal features processed (old files were cleared).")

Done: 230 categorical/ordinal features processed (old files were cleared).
